In [ ]:
!pip install -q --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "datasets<4.4.0" "trl<=0.24.0"


In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets =UserSecretsClient()
    hf_token=user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("logged in the hugging face account using the token")
except Exception as e:
    print("Not able to log in")

In [ ]:
from unsloth import FastLanguageModel # to import the model (mistral)
from datasets import load_dataset  # to import our dataset from  hf
from unsloth.chat_templates import get_chat_template # this is used to tokenise the message
# chatml is used like a set of rules which helps the llm to understand which part of message is the query by user and 
# and which part is the solution to the query using delimiter <start>system<end><start>user:....<end><start>user:<end>
# basically a tokeniser but not fully tokeninsing our message

In [ ]:
# calling the specefic llm
model,tokenizer=FastLanguageModel.from_pretrained(
    model_name="GAnirudh711/mistral-7b-dsa-expert",
    max_seq_length=2048,
    # max_seq_length=4096 means we have 4096 tokens (ie 3k words can be handled).token!=word btw.
    
    dtype=None,
    # datatype is none as we don't concern ourselves with the datatype of the dataset (auto handled)
    
    load_in_4bit=True,# we are calling the 4bit instead of the std. 16bits as it will be correct on the kaggle nb
    # otherwise the kaggle will not be able to handle the 16 bit gpu memory processing
)

In [ ]:
tokenizer=get_chat_template(
    tokenizer,
    chat_template="chatml",
)

In [ ]:
def format_for_ai(row):
    row["text"]=tokenizer.apply_chat_template(
        row["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return row
print("loading the dataset we created from hugging face\n")
dataset_path="GAnirudh711/DSA_FINETUNING_LLM_V2"
mega_dataset=load_dataset(dataset_path,split="train")

print("converting the dataset in the required format using map\n")
mega_dataset=mega_dataset.map(format_for_ai,num_proc=2)
mega_dataset = mega_dataset.map(lambda r: {"n_tok": len(tokenizer(r["text"]).input_ids)})
print("share over 2048 tokens:", sum(n > 2048 for n in mega_dataset["n_tok"]) / len(mega_dataset))
mega_dataset = mega_dataset.filter(lambda r: r["n_tok"] <= 2048)
print("converted the initial dataet in the correct format\n")
print(f'the length is {len(mega_dataset)}')


In [ ]:
FastLanguageModel.for_training(model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = model.num_parameters()
print(f"Trainable: {trainable:,} ({100*trainable/total:.2f}%)")


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
import torch

MARKER = "<|im_start|>assistant\n"

def tokenize_and_mask(row):
    text = row["text"]
    idx = text.rfind(MARKER)
    if idx == -1:
        return {"input_ids": [], "attention_mask": [], "labels": []}
    prompt_text = text[: idx + len(MARKER)]
    full_ids = tokenizer(text, add_special_tokens=True)["input_ids"]
    n_prompt = len(tokenizer(prompt_text, add_special_tokens=True)["input_ids"])
    labels = [-100] * n_prompt + full_ids[n_prompt:]
    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels[: len(full_ids)],
    }

train_ds = mega_dataset.map(tokenize_and_mask, remove_columns=mega_dataset.column_names)
train_ds = train_ds.filter(lambda r: len(r["input_ids"]) > 0 and any(l != -100 for l in r["labels"]))
print("training rows:", len(train_ds))

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    max_seq_length=2048,
    packing=False,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, label_pad_token_id=-100),

    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        warmup_steps=10,
        learning_rate=1e-4,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        optim="adamw_8bit",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_steps=100,
        save_total_limit=2,
        output_dir="outputs_v2",
        report_to="none",
        remove_unused_columns=False,
        average_tokens_across_devices=False,
        seed=42,
    ),
)

print("\n STARTING TRAINING!")
trainer_stats = trainer.train()
print("\n TRAINING COMPLETE!")

model.save_pretrained("v2_adapter")
tokenizer.save_pretrained("v2_adapter")
print("Saved to v2_adapter")

# upload automatically so you do not need to be present
from huggingface_hub import HfApi
api = HfApi()
repo_id = "GAnirudh711/mistral-7b-dsa-expert-v2"
api.create_repo(repo_id, exist_ok=True)
api.upload_folder(folder_path="v2_adapter", repo_id=repo_id, repo_type="model")
print("UPLOADED:", f"https://huggingface.co/{repo_id}")